In [1]:
import os 
os.chdir("../")
%pwd

'd:\\Programming\\ML\\End-to-End\\End-to-End-TelcoChurn'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str 
    data_dir: Path 
    all_schemas: dict

In [3]:
from src.utils import read_yaml, create_directories
from src.constants import *
from pathlib import Path
import pandas as pd

class ConfigurationManager:
    def __init__(self,
                config:Path = Path(CONFIG_FILE_PATH),
                params:Path = Path(PARAMS_FILE_PATH),
                schema:Path = Path(SCHEMA_FILE_PATH)):
        self.config = read_yaml(config)
        self.params = read_yaml(params)
        self.schema = read_yaml(schema)
        create_directories([self.config.artifacts_root])
        
    def get_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation 
        schema = self.schema.COLUMNS 
        
        create_directories([config.root_dir])
        
        return DataValidationConfig(
            root_dir = Path(config.root_dir),
            STATUS_FILE = config.STATUS_FILE, 
            data_dir = Path(config.data_dir),
            all_schemas = schema   
        )

In [13]:
import pandas as pd
from src import CustomException, logging

class DataValidation:
    def __init__(self, config):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            df = pd.read_csv(self.config.data_dir).drop(columns="customerID", errors="ignore")
            schema = self.config.all_schemas  # dict from YAML

            # --- 1. Validate column names ---
            dataset_cols = set(df.columns)
            schema_cols = set(schema.keys())

            missing_cols = schema_cols - dataset_cols
            extra_cols = dataset_cols - schema_cols

            if missing_cols:
                logging.error(f"Missing columns in data: {missing_cols}")
            if extra_cols:
                logging.warning(f"Extra columns not in schema: {extra_cols}")

            # --- 2. Validate data types ---
            dtype_mismatch = {}
            for col, expected_dtype in schema.items():
                if col in df.columns:
                    actual_dtype = str(df[col].dtype)
                    if actual_dtype != expected_dtype:
                        dtype_mismatch[col] = {'expected': expected_dtype, 'found': actual_dtype}

            validation_status = not (missing_cols or dtype_mismatch)

            # --- 3. Write status file ---
            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}\n")
                if missing_cols:
                    f.write(f"Missing columns: {missing_cols}\n")
                if dtype_mismatch:
                    f.write(f"Dtype mismatches: {dtype_mismatch}\n")

            logging.info(f"Validation complete. Status: {validation_status}")
            return validation_status

        except Exception as e:
            raise CustomException(e)


In [14]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise CustomException(e)

[2025-10-15 09:46:39,975] [INFO] [root:read_yaml:16] - reading the content of 'config\config.yaml'
[2025-10-15 09:46:39,977] [INFO] [root:read_yaml:16] - reading the content of 'params.yaml'
[2025-10-15 09:46:39,990] [INFO] [root:read_yaml:16] - reading the content of 'schema.yaml'
[2025-10-15 09:46:39,991] [INFO] [root:create_directories:39] - created directory at: artifacts
[2025-10-15 09:46:39,992] [INFO] [root:create_directories:39] - created directory at: artifacts/data_validation
[2025-10-15 09:46:40,024] [INFO] [root:validate_all_columns:43] - Validation complete. Status: True
